This notebook contains the weather data preprocessing. By commenting lines in/out, you can adjust whether to work on the real-world data or the synthetic data.

In [2]:
# imports
import pandas as pd
from pathlib import Path
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

sns.set_style("whitegrid")

In [ ]:
### load and clean data
### skip this for synthetic data creation
df = pd.read_csv("beijing_pm25.csv")

# drop rows with missing target variable
df = df.dropna(subset=["pm2.5"]).reset_index(drop=True)

# define time index
df["timestamp"] = pd.to_datetime(
    df[["year", "month", "day", "hour"]]
)
df = df.sort_values("timestamp").reset_index(drop=True)

# define columns to retain
cols_keep = [
    "timestamp",
    "pm2.5", "DEWP", "TEMP", "PRES",
    "Iws", "Is", "Ir", "cbwd"
]

df = df[cols_keep]

In [4]:
# save pruned datasets with different clipping strengths

# output directory
# out_dir = Path("data/real_pruned")
# use this for synthetic data creation instead
df = pd.read_csv("data/synthetic_pruned/clip_alpha_0.000.csv")
out_dir = Path("data/synthetic_pruned")
out_dir.mkdir(parents=True, exist_ok=True)

# features eligible for pruning (explicitly excludes pm2.5)
prune_features = ["DEWP", "TEMP", "PRES", "Iws", "Is", "Ir"]

# pruning strengths
alphas = [0.0, 0.005, 0.01, 0.02, 0.05]


def quantile_clip_raw(df, features, alpha):
    df_out = df.copy()
    for col in features:
        lo = df[col].quantile(alpha)
        hi = df[col].quantile(1 - alpha)
        df_out[col] = df[col].clip(lo, hi)
    return df_out

for alpha in alphas:
    df_pruned = quantile_clip_raw(df, prune_features, alpha)
    # save
    fname = out_dir / f"clip_alpha_{alpha:.3f}.csv"
    df_pruned.to_csv(fname, index=False)

    print(f"Saved {fname}  | rows={len(df_pruned)}")

Saved data\synthetic_pruned\clip_alpha_0.000.csv  | rows=41757
Saved data\synthetic_pruned\clip_alpha_0.005.csv  | rows=41757
Saved data\synthetic_pruned\clip_alpha_0.010.csv  | rows=41757
Saved data\synthetic_pruned\clip_alpha_0.020.csv  | rows=41757
Saved data\synthetic_pruned\clip_alpha_0.050.csv  | rows=41757
